# 01 - Corpus exploration

Measures the property that motivates the whole architecture: the three
branches hold equal numbers of documents but contribute very unequal
numbers of retrievable chunks.

It also verifies the cleaning pipeline removed every escape and entity
artifact found in the raw datasets.

In [1]:
import sys
from pathlib import Path

CAPSTONE = Path(r"C:\Users\modij\Downloads\MUJ-DS-23FE10CDS00439\capstone")
sys.path.insert(0, str(CAPSTONE / "src"))

from dotenv import load_dotenv
load_dotenv(CAPSTONE / ".env")

import pandas as pd
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 70)

In [2]:
from branched_rag.config import load_config
from branched_rag.data import load_corpus, load_chunks, summarise

config = load_config(CAPSTONE / 'config.yaml')
corpus = load_corpus(config)
chunks = load_chunks(config)

print(f'documents: {len(corpus):,}')
print(f'chunks:    {len(chunks):,}')

C:\Users\modij\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


documents: 6,000
chunks:    16,641


## Branch imbalance

Equal documents in, very unequal chunks out. This is the measurement the
design rests on: pooled into one index, the largest branch dominates
top-k on volume alone.

In [3]:
summarise(chunks).as_frame()

,branch,documents,chunks,mean_chunk_chars
0,reviews,2000,4436,233.9
1,news,2000,2040,226.3
2,financials,2000,10165,539.4


In [4]:
share = chunks.branch.value_counts(normalize=True).round(3)
print('share of all retrievable chunks:')
print(share.to_string())
print()
print(f'largest branch holds {share.max():.1%} of the corpus')

share of all retrievable chunks:
branch
financials    0.611
reviews       0.267
news          0.123

largest branch holds 61.1% of the corpus


## Document and chunk length by branch

In [5]:
stats = (chunks.groupby('branch', sort=False)
         .agg(chunks=('chunk_id', 'count'),
              docs=('doc_id', 'nunique'),
              mean_chars=('n_chars', 'mean'),
              median_chars=('n_chars', 'median'),
              max_chars=('n_chars', 'max'))
         .round(1))
stats['chunks_per_doc'] = (stats.chunks / stats.docs).round(2)
stats

,chunks,docs,mean_chars,median_chars,max_chars,chunks_per_doc
branch,,,,,,
reviews,4436,2000,233.9,247.0,939,2.22
news,2040,2000,226.3,215.0,783,1.02
financials,10165,2000,539.4,533.0,13066,5.08


`chunks_per_doc` is the mechanism. Financial articles are long, so each
document becomes several retrievable units, while a news item becomes
about one.

## Cleaning verification

No chunk may retain a raw-dataset artifact.

In [6]:
patterns = {
    'lone backslash': r'\\\\',
    'entity fragment (#39;)': r'#\\d+;',
    'unescaped &amp;': r'&amp;',
    'numeric entity': r'&#',
}
for name, pattern in patterns.items():
    hits = chunks.text.str.contains(pattern, regex=True, na=False).sum()
    print(f'{name:28s} {hits:>6,} chunks')

lone backslash                    0 chunks
entity fragment (#39;)            0 chunks
unescaped &amp;                   0 chunks
numeric entity                    0 chunks


## Review branch polarity balance

An earlier version of the admission filter used valence-carrying cues
(`defective`, `flimsy`, `overpriced`) and produced a corpus that was 67%
negative, which would have biased every sentiment aggregate. The filter
now uses neutral product cues and the sample is balanced at ingest.

In [7]:
import json

reviews = corpus[corpus.branch == 'reviews']
polarity = reviews.extra.map(lambda e: json.loads(e)['stated_polarity'])
print(polarity.value_counts().to_string())
print()
print(polarity.value_counts(normalize=True).round(3).to_string())

extra
negative    1000
positive    1000

extra
negative    0.5
positive    0.5


## Sample documents

In [8]:
for branch in ['reviews', 'news', 'financials']:
    row = corpus[corpus.branch == branch].iloc[7]
    print(f'=== {branch} | {row.title[:70]}')
    print(row.text[:300].strip())
    print()

=== reviews | Poor instructions, but once you figure it out. kinda cool
These wraps were not doing anything when we placed them in "hot" water. We searched the internet and found others having the same problem. The instructions are very poor. First off, you need to dip in boiling water (or at least some temperature level above what I would call hot). Second, the instruc

=== news | Fed minutes show dissent over inflation (USATODAY.com) USATODAY.com
Retail sales bounced back a bit in July, and new claims for jobless benefits fell last week, the government said Thursday, indicating the economy is improving from a midsummer slump.

=== financials | Oil dips away from 2014 highs on rising U.S. rig count, but analysts s
January 15, 2018 / 12:41 AM / Updated an hour ago Oil hovers near three-year high despite rising U.S. output Nia Williams , Rod Nickel 3 Min Read CALGARY, Alberta (Reuters) - Oil hovered near a three-year high above $70 a barrel on Monday on signs that production cuts by OPE